In [ ]:
# CELL 1 — Install libraries and import everything
!pip install timm tqdm kaggle

import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.datasets import ImageFolder
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import timm
from tqdm import tqdm
import os
import shutil
import matplotlib.pyplot as plt

In [ ]:
# CELL 2 — Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# CELL 3 — Unzip the uploaded archive
!unzip -q '/content/archive (4).zip' -d /content/extracted

print("Extracted ✅")
!ls /content/extracted/
!ls /content/extracted/bean_dataset/

In [ ]:
# CELL 4 — Check the dataset structure
import os

dataset_root = '/content/extracted/Bean_Dataset'

for cls in os.listdir(dataset_root):
    cls_path = os.path.join(dataset_root, cls)
    if os.path.isdir(cls_path):
        count = len(os.listdir(cls_path))
        print(f"└── {cls}: {count} images")

In [ ]:
#CELL 5
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split

# Load full dataset
full_dataset = ImageFolder('/content/extracted/Bean_Dataset')

print("Classes:", full_dataset.classes)
print("Total samples:", len(full_dataset))

# Split into train/val manually (80/20)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

print("Train samples:", train_size)
print("Val samples:", val_size)

In [ ]:
# CELL 6 — Define transforms for two resolutions
# Bean images are real-world photos — 224x224 is better than 64x64 for pretrained models

def get_transforms():
    return [
        # High resolution — what the model ideally receives
        transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ]),
        # Low resolution — simulates a cheap/compressed image
        transforms.Compose([
            transforms.Resize((112, 112)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])
    ]

In [ ]:
# CELL 7 — Custom dataset that returns multiple resolutions of the same image

class MultiResDataset(Dataset):
    def __init__(self, dataset, transforms_list):
        self.dataset = dataset
        self.transforms_list = transforms_list

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]

        # Apply each transform to the SAME image
        imgs = [t(img) for t in self.transforms_list]

        return imgs, label

In [ ]:
#CELL 8 -Wrap with MultiResDataset and create DataLoaders
transforms_list = get_transforms()

train_data = MultiResDataset(train_dataset, transforms_list)
val_data = MultiResDataset(val_dataset, transforms_list)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=2)

print("Loaders ready ✅")

In [ ]:
# CELL 9 — Load pretrained ResNet18 and adapt for 3 classes

model = timm.create_model('resnet18', pretrained=True, num_classes=3)
model = model.to(device)

print("Model architecture: ResNet18")
print("Output classes: 3 (Angular Leaf Spot, Bean Rust, Healthy)")
print("Pretrained weights: ImageNet ✅")
print("Model ready ✅")

In [ ]:
# CELL 10 — Define loss function and optimizer

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print("Loss: CrossEntropyLoss")
print("Optimizer: Adam (lr=0.0001)")

In [ ]:
# CELL 11 — KL Divergence consistency loss
# Forces the model to give similar predictions for high-res and low-res versions

def consistency_loss(outputs):
    loss = 0
    count = 0

    for i in range(len(outputs)):
        for j in range(i+1, len(outputs)):
            loss += F.kl_div(
                F.log_softmax(outputs[i], dim=1),
                F.softmax(outputs[j], dim=1),
                reduction='batchmean'
            )
            count += 1

    return loss / count

In [ ]:
# CELL 12 — Training loop function

def train(model, loader):
    model.train()
    total_loss = 0

    for imgs, labels in tqdm(loader):
        labels = labels.to(device)

        outputs = []
        for img in imgs:
            img = img.to(device)
            outputs.append(model(img))

        # Classification loss: model should predict the RIGHT class
        ce = criterion(outputs[0], labels)

        # Consistency loss: high-res and low-res predictions should MATCH
        cons = consistency_loss(outputs)

        # Combined loss
        loss = ce + 0.5 * cons

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
# CELL 13 — Train for 10 epochs and track loss

loss_history = []
epochs = 20

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    loss = train(model, train_loader)
    print(f"Loss: {loss:.4f}")
    loss_history.append(loss)

print("\nTraining complete ✅")

In [ ]:
# CELL 14 — Evaluate accuracy at both resolutions

def evaluate(model, loader, res_idx=0):
    """res_idx=0 for high-res (224x224), res_idx=1 for low-res (112x112)"""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for imgs, labels in loader:
            labels = labels.to(device)
            outputs = model(imgs[res_idx].to(device))
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return 100 * correct / total

high_res_acc = evaluate(model, val_loader, res_idx=0)
low_res_acc = evaluate(model, val_loader, res_idx=1)

print(f"High-Res (224x224) Accuracy: {high_res_acc:.2f}%")
print(f"Low-Res (112x112) Accuracy: {low_res_acc:.2f}%")

In [ ]:
# CELL 15 — Save model

torch.save(model.state_dict(), "bean_model.pth")
print("Model saved locally ✅")

# Optional: save to Google Drive
from google.colab import drive
drive.mount('/content/drive')
torch.save(model.state_dict(), "/content/drive/MyDrive/bean_model.pth")
print("Saved to Drive ✅")

In [ ]:
# CELL 16 — Plot training loss

plt.figure()
plt.plot(range(1, len(loss_history)+1), loss_history, marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training Loss vs Epochs")
plt.grid()
plt.savefig("loss_graph.png")
plt.show()

# Accuracy bar chart
plt.figure()
labels_plot = ['High-Res (224x224)', 'Low-Res (112x112)']
values_plot = [high_res_acc, low_res_acc]
plt.bar(labels_plot, values_plot, color=['steelblue', 'coral'])
plt.ylabel("Accuracy (%)")
plt.title("Accuracy vs Resolution — Bean Disease")
plt.savefig("accuracy_graph.png")
plt.show()

In [ ]:
# CELL 17 — Unzip Mendeley dataset
!unzip -q '/content/beans-20251022T043712Z-1-001.zip' -d /content/mendeley_test

# Check what's inside
!find /content/mendeley_test -type d

In [ ]:
# CELL 18 — Load Mendeley dataset

mendeley_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

mendeley_dataset = ImageFolder('/content/mendeley_test/beans', transform=mendeley_transform)
print("Mendeley classes found:", mendeley_dataset.classes)
print("Class to index:", mendeley_dataset.class_to_idx)
print("Total samples:", len(mendeley_dataset))

In [ ]:
# CELL 19 — Test model on Mendeley unseen data

from torch.utils.data import DataLoader

mendeley_loader = DataLoader(mendeley_dataset, batch_size=32, shuffle=False)

# Mendeley: bacterial_pathogen=0, no_disease=1
# Bean model: angular_leaf_spot=0, healthy=2
mendeley_to_bean = {
    0: 0,  # bacterial_pathogen → angular_leaf_spot
    1: 2   # no_disease → healthy
}

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for imgs, labels in mendeley_loader:
        imgs = imgs.to(device)

        remapped_labels = torch.tensor(
            [mendeley_to_bean[l.item()] for l in labels]
        ).to(device)

        outputs = model(imgs)
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == remapped_labels).sum().item()
        total += remapped_labels.size(0)

unseen_acc = 100 * correct / total
print(f"\nUnseen Dataset (Mendeley) Accuracy: {unseen_acc:.2f}%")
print(f"Tested on {total} samples")

In [ ]:
# Run this BEFORE Cell 13 — reset model to fresh pretrained weights
import timm
model = timm.create_model('resnet18', pretrained=True, num_classes=3)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0001)
print("Model reset ✅")